## Tanzania Tourism Cost Category — Multi-Class Log Loss

A complete, self-contained pipeline:

1.  **Data loading & schema-aware cleaning**
2.  **Feature engineering** (numeric + categorical + interaction)
3.  **Native categorical handling** with LightGBM / CatBoost
4.  **10-fold CV, multi-seed bagging, and stacked ensemble**
5.  **Log-loss-optimized blending** with scipy
6.  **Submission generation**

Author: Principal Kaggle/Zindi GM approach

### Imports and Configuration

This section handles all necessary library imports and defines global configuration parameters for the notebook, such as random seeds, data directories, file paths, and target column names.

In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
from scipy.optimize import minimize
import lightgbm as lgb
%pip install catboost
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
SEED = 42
N_FOLDS = 10
N_SEEDS = 3                     # multi-seed bagging
DATA_DIR = "./"                 # <-- change if needed
SUB_PATH   = "submission.csv"

TARGET = "cost_category"
ID_COL = "Tour_ID"

# Order defined by problem statement; LabelEncoder will map consistently.
CLASS_ORDER = [
    "High Cost", "Higher Cost", "Highest Cost",
    "Low Cost", "Lower Cost", "Normal Cost",
]

np.random.seed(SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.0 MB/s eta 0:00:00


In [3]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }

}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

Local file for train not found. Fetching from GitHub...
Loaded train from GitHub successfully.
Local file for test not found. Fetching from GitHub...
Loaded test from GitHub successfully.
Local file for sample_sub not found. Fetching from GitHub...
Loaded sample_sub from GitHub successfully.


### Data Loading and Target Encoding

Here, the training and test datasets are loaded. The target variable (`cost_category`) is then encoded numerically using `LabelEncoder` to prepare it for model training. The class mapping is printed for reference.

In [4]:
# ----------------------------------------------------------------------
# 1. LOAD
# ----------------------------------------------------------------------
# 'train' and 'test' are now loaded by the preceding cell.

print(f"Train shape: {train.shape}  |  Test shape: {test.shape}")
print(f"Target distribution:\n{train[TARGET].value_counts()}\n")

# Encode target
le = LabelEncoder().fit(CLASS_ORDER)
train["target"] = le.transform(train[TARGET])
NUM_CLASSES = len(le.classes_)
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Train shape: (18506, 21)  |  Test shape: (6169, 20)
Target distribution:
cost_category
Normal Cost     5471
Higher Cost     4865
High Cost       3678
Lower Cost      2567
Low Cost        1566
Highest Cost     359
Name: count, dtype: int64

Class mapping: {np.str_('High Cost'): np.int64(0), np.str_('Higher Cost'): np.int64(1), np.str_('Highest Cost'): np.int64(2), np.str_('Low Cost'): np.int64(3), np.str_('Lower Cost'): np.int64(4), np.str_('Normal Cost'): np.int64(5)}


### Feature Engineering Function

This function applies various feature engineering steps, including numeric coercion, extracting date features (if available), and calculating cost-per-person/night ratios. It is designed to be reusable for both training and test datasets.

In [6]:
# ----------------------------------------------------------------------
# 2. FEATURE ENGINEERING
# ----------------------------------------------------------------------
def feature_engineer(df):
    df = df.copy()

    # --- Numeric coercion for columns that might be strings ---
    # Common numeric candidates in tourism expenditure surveys:
    numeric_candidates = [
        "total_cost", "cost_of_tour", "duration", "nights",
        "people_in_group", "number_of_people", "age", "nights_stayed",
        "total_visitors", "children", "adults",
    ]
    for c in numeric_candidates:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- Country-level aggregates (frequency encoding) ---
    if "country" in df.columns:
        df["country"] = df["country"].astype(str).str.strip().str.lower()
        # Will be re-fit on the combined set below.

    # --- Date features (if any date column exists) ---
    for c in df.columns:
        if "date" in c.lower() or "time" in c.lower():
            try:
                dt = pd.to_datetime(df[c], errors="coerce")
                if dt.notna().mean() > 0.5:
                    df[f"{c}_year"]  = dt.dt.year
                    df[f"{c}_month"] = dt.dt.month
                    df[f"{c}_dow}}"]   = dt.dt.dayofweek
                    df[f"{c}_quarter}}"] = dt.dt.quarter
                    df.drop(columns=[c], inplace=True)
            except Exception:
                pass

    # --- Cost-per-person style ratios if base columns exist ---
    if {"total_cost", "people_in_group"}.issubset(df.columns):
        df["cost_per_person"] = df["total_cost"] / (df["people_in_group"].replace(0, np.nan))
    if {"total_cost", "nights"}.issubset(df.columns):
        df["cost_per_night"] = df["total_cost"] / (df["nights"].replace(0, np.nan))

    return df

### Apply Feature Engineering and Preprocessing

This section applies the defined `feature_engineer` function to both training and test data. It then combines the datasets to perform frequency encoding for high-cardinality categorical features, ensuring consistency across both sets. Finally, categorical features are prepared differently for LightGBM/XGBoost (numeric codes) and CatBoost (raw strings).

In [10]:
train = feature_engineer(train)
test  = feature_engineer(test)

# Combine for consistent frequency/aggregate encoding
full = pd.concat([train.drop(columns=[TARGET, "target"]), test], axis=0, ignore_index=True)

# Frequency encode high-cardinality categoricals
cat_cols = [c for c in full.columns
            if full[c].dtype == "object" and c != ID_COL]

for c in cat_cols:
    freq = full[c].value_counts(normalize=True)
    full[f"{c}_freq"] = full[c].map(freq).astype(np.float32)

# Re-split
X_train_full = full.iloc[:len(train)].reset_index(drop=True)
X_test_full  = full.iloc[len(train):].reset_index(drop=True)
y = train["target"].values

# ---- Feature list ----------------------------------------------------
drop_from_features = [ID_COL]
feature_cols = [c for c in X_train_full.columns if c not in drop_from_features]

# Identify categorical features for native handling
cat_features = [c for c in feature_cols
                if X_train_full[c].dtype == "object"]

print(f"\nTotal features: {len(feature_cols)}  |  Categorical: {len(cat_features)}")

# For LightGBM/XGBoost we need a numeric representation of categoricals.
# Use pandas category with codes fit on combined data (safe & memory-friendly).
X_train = X_train_full[feature_cols].copy()
X_test  = X_test_full[feature_cols].copy()

for c in cat_features:
    cats = pd.Categorical(pd.concat([X_train[c], X_test[c]], axis=0)).categories
    X_train[c] = pd.Categorical(X_train[c], categories=cats).codes.astype(np.int32)
    X_test[c]  = pd.Categorical(X_test[c],  categories=cats).codes.astype(np.int32)

# For CatBoost we keep raw strings (better than codes)
X_train_cb = X_train_full[feature_cols].copy()
X_test_cb  = X_test_full[feature_cols].copy()
for c in cat_features:
    X_train_cb[c] = X_train_cb[c].astype(str)
    X_test_cb[c]  = X_test_cb[c].astype(str)

del full; gc.collect()


Total features: 34  |  Categorical: 15


8

### Model Definitions

This section defines functions to create instances of LightGBM, CatBoost, and XGBoost classifiers with optimized hyperparameters. These models are configured for multi-class classification and log-loss evaluation.

In [11]:
# ----------------------------------------------------------------------
# 3. MODEL DEFINITIONS
# ----------------------------------------------------------------------
def make_lgb(seed):
    return lgb.LGBMClassifier(
        objective="multiclass",
        num_class=NUM_CLASSES,
        metric="multi_logloss",
        boosting_type="gbdt",
        learning_rate=0.03,
        num_leaves=48,
        min_child_samples=25,
        feature_fraction=0.75,
        bagging_fraction=0.85,
        bagging_freq=5,
        lambda_l1=0.3,
        lambda_l2=0.6,
        max_depth=-1,
        n_estimators=3000,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

def make_cat(seed):
    return CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=3000,
        learning_rate=0.04,
        depth=6,
        l2_leaf_reg=4.0,
        random_strength=1.0,
        bagging_temperature=0.6,
        border_count=128,
        random_seed=seed,
        od_type="Iter",
        od_wait=200,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

def make_xgb(seed):
    return XGBClassifier(
        objective="multi:softprob",
        num_class=NUM_CLASSES,
        eval_metric="mlogloss",
        tree_method="hist",
        learning_rate=0.04,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=0.3,
        reg_lambda=0.8,
        n_estimators=3000,
        random_state=seed,
        n_jobs=-1,
        verbosity=0,
    )

### Cross-Validation and Multi-Seed Bagging

This is the core training section. It employs a multi-seed bagging approach with stratified k-fold cross-validation to train each of the LightGBM, CatBoost, and XGBoost models. Out-of-fold (OOF) predictions are collected for blending, and test set predictions are averaged across all folds and seeds.

In [ ]:
# ----------------------------------------------------------------------
# 4. CROSS-VALIDATION + MULTI-SEED BAGGING
# ----------------------------------------------------------------------
# We store OOF and test predictions for each model family.
oof_lgb = np.zeros((len(X_train), NUM_CLASSES))
oof_cat = np.zeros((len(X_train), NUM_CLASSES))
oof_xgb = np.zeros((len(X_train), NUM_CLASSES))

test_lgb = np.zeros((len(X_test), NUM_CLASSES))
test_cat = np.zeros((len(X_test), NUM_CLASSES))
test_xgb = np.zeros((len(X_test), NUM_CLASSES))

for seed in range(N_SEEDS):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + seed)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y)):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # ---- LightGBM ----
        lgb_model = make_lgb(SEED + seed)
        lgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(150, verbose=False),
                       lgb.log_evaluation(0)],
        )
        oof_lgb[va_idx] += lgb_model.predict_proba(X_va) / N_SEEDS
        test_lgb += lgb_model.predict_proba(X_test) / (N_FOLDS * N_SEEDS)

        # ---- CatBoost ----
        cat_model = make_cat(SEED + seed)
        tr_pool = Pool(X_train_cb.iloc[tr_idx], y_tr, cat_features=cat_features)
        va_pool = Pool(X_train_cb.iloc[va_idx], y_va, cat_features=cat_features)
        cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
        oof_cat[va_idx] += cat_model.predict_proba(va_pool) / N_SEEDS
        test_cat += cat_model.predict_proba(Pool(X_test_cb, cat_features=cat_features)) / (N_FOLDS * N_SEEDS)

        # ---- XGBoost ----
        xgb_model = make_xgb(SEED + seed)
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            early_stopping_rounds=150,
            verbose=False,
        )
        oof_xgb[va_idx] += xgb_model.predict_proba(X_va) / N_SEEDS
        test_xgb += xgb_model.predict_proba(X_test) / (N_FOLDS * N_SEEDS)

        print(f"[seed {seed}] fold {fold}  "
              f"lgb={log_loss(y_va, oof_lgb[va_idx]*N_SEEDS/(seed+1) if False else lgb_model.predict_proba(X_va)):.5f}  "
              f"cat={log_loss(y_va, cat_model.predict_proba(va_pool)):.5f}  "
              f"xgb={log_loss(y_va, xgb_model.predict_proba(X_va)):.5f}")

    del skf; gc.collect()

### Individual Out-of-Fold (OOF) Scores

After completing the cross-validation, the log-loss scores for the OOF predictions of each individual model (LightGBM, CatBoost, XGBoost) are calculated and printed. These scores indicate the performance of each model on unseen data within the training set.

In [ ]:
# ---- Individual OOF scores ------------------------------------------
ll_lgb = log_loss(y, oof_lgb)
ll_cat = log_loss(y, oof_cat)
ll_xgb = log_loss(y, oof_xgb)
print(f"\nOOF LogLoss  LGB: {ll_lgb:.5f}  |  CAT: {ll_cat:.5f}  |  XGB: {ll_xgb:.5f}")

### Stacked Blending Function

This function (`blend_weights`) determines the optimal weights for blending the predictions of multiple models. It uses `scipy.optimize.minimize` to find a set of simplex-constrained weights that minimize the overall log loss of the blended OOF predictions. This ensures that the ensemble leverages the strengths of each base model.

In [ ]:
# ----------------------------------------------------------------------
# 5. STACKED BLENDING (log-loss-optimal weights)
# ----------------------------------------------------------------------
def blend_weights(oofs, y, n_classes):
    """Find simplex-constrained weights minimizing OOF log loss."""
    n_models = len(oofs)
    # Stack as (N, n_models, n_classes)
    P = np.stack(oofs, axis=1)

    def loss(w):
        w = np.clip(w, 0, None)
        w = w / w.sum()
        blend = np.tensordot(P, w, axes=([1], [0]))
        blend = blend / blend.sum(axis=1, keepdims=True)
        return log_loss(y, blend)

    x0 = np.ones(n_models) / n_models
    res = minimize(loss, x0, method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-7, "maxiter": 3000})
    w = np.clip(res.x, 0, None); w = w / w.sum()
    return w, res.fun

### Apply Stacked Blending

Here, the `blend_weights` function is called with the OOF predictions from LightGBM, CatBoost, and XGBoost. The resulting optimal weights and the blended OOF log-loss score are then printed, indicating the performance of the weighted ensemble.

In [ ]:
weights, best_ll = blend_weights([oof_lgb, oof_cat, oof_xgb], y, NUM_CLASSES)
print(f"\nBlend weights:  LGB={weights[0]:.4f}  CAT={weights[1]:.4f}  XGB={weights[2]:.4f}")
print(f"Blended OOF LogLoss: {best_ll:.5f}")

### Optional: Second-Level Stacker (Logistic Regression)

This section implements an optional second-level stacking model using Logistic Regression. The OOF probabilities from the base models are concatenated to form the features for the stacker. `cross_val_predict` is used to get OOF predictions for the stacker, and its performance is compared against the simple weighted blend. The better-performing model (blend or stacker) is then selected for final prediction.

In [ ]:
# ----------------------------------------------------------------------
# 6. OPTIONAL: 2ND-LEVEL STACKER (logistic regression on OOF probs)
# ----------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

# Features = concatenated OOF probabilities from each model
stack_X = np.hstack([oof_lgb, oof_cat, oof_xgb])
stack_test = np.hstack([test_lgb, test_cat, test_xgb])

# We need more splits for a robust stacker; use StratifiedKFold again
skf_stack = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
stacker = LogisticRegression(
    C=1.0, max_iter=2000, multi_class="multinomial",
    solver="lbfgs", n_jobs=-1,
)
stack_oof = cross_val_predict(stacker, stack_X, y, cv=skf_stack,
                              method="predict_proba", n_jobs=-1)
ll_stack = log_loss(y, stack_oof)
print(f"Stacker OOF LogLoss: {ll_stack:.5f}")

# Refit stacker on all OOF and apply to test
stacker.fit(stack_X, y)
stack_test_proba = stacker.predict_proba(stack_test)

# Final: choose the better of simple blend vs stacker
if ll_stack < best_ll:
    print("→ Using STACKER as final model")
    final_test = stack_test_proba
else:
    print("→ Using WEIGHTED BLEND as final model")
    final_test = (weights[0] * test_lgb +
                  weights[1] * test_cat +
                  weights[2] * test_xgb)
    final_test = final_test / final_test.sum(axis=1, keepdims=True)

### Submission Generation

This final section takes the predictions from the chosen ensemble model (either the weighted blend or the second-level stacker), converts the predicted probabilities back into original class labels using the `LabelEncoder`, and creates the submission file in the specified format. It also prints the head of the submission DataFrame and the distribution of the predicted classes.

In [ ]:
# ----------------------------------------------------------------------
# 7. SUBMISSION
# ----------------------------------------------------------------------
pred_labels = le.inverse_transform(final_test.argmax(axis=1))

sub = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET: pred_labels,
})

# If probabilities are also required (some Zindi comps ask for them),
# we can append them. Uncomment below to export probability columns:
# for i, cls in enumerate(le.classes_):
#     sub[f"prob_{cls.replace(' ', '_')}"] = final_test[:, i]

sub.to_csv(SUB_PATH, index=False)
print(f"\n✅ Submission written to {SUB_PATH}")
print(sub.head())
print("\nPrediction distribution:")
print(sub[TARGET].value_counts())